In [13]:
#!pip install gudhi

In [14]:
import os
import tqdm
import re
import numpy as np
import pandas as pd
import seaborn as sns
import gudhi as gd
import gudhi.representations
import matplotlib.pyplot as plt
from xgboost                 import XGBClassifier
from sklearn.preprocessing   import MinMaxScaler
from sklearn.pipeline        import Pipeline, FeatureUnion
from sklearn.svm             import SVC
from sklearn.ensemble        import RandomForestClassifier
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics         import accuracy_score, precision_score, recall_score, f1_score
from sklearn.base            import BaseEstimator, TransformerMixin

In [15]:
# Persistence diagram loader
def load_persistence_diagram(file):
    df = pd.read_csv(file, sep="\t", header=None)
    df.columns = ["dim", "birth", "death", "pair_info"]
    formatted = [(row["dim"], (row["birth"], row["death"])) for _, row in df.iterrows()]
    return [pair for dim, pair in formatted if dim == 1]

def get_files_from_folder(folder_path):
    """Reads all files from a folder based on specified rules."""
    files = []
    for filename in os.listdir(folder_path):
        # Exclude files ending with "_persisdiagram.txt" that contain "_edgeperm_", "_randperm_", "_distperm_", or "_chrX_"
        if filename.endswith("_persisdiagram.txt") and "_edgeperm_" not in filename and "_randperm_" not in filename and "_distperm_" not in filename and "_chrX_" not in filename:
            files.append(os.path.join(folder_path, filename))
    return files

def calculate_persistent_entropy(diagram):
    """Calculates the Persistent Entropy of a persistence diagram."""
    persistences = np.array([death - birth for birth, death in diagram])
    # Filter out points with zero persistence
    non_zero_persistences = persistences[persistences > 0]

    if len(non_zero_persistences) == 0:
        return 0.0  # Return 0 if all persistences are zero

    total_persistence = np.sum(non_zero_persistences)
    # Avoid division by zero if total_persistence is 0 (though already handled by non_zero_persistences check)
    if total_persistence == 0:
        return 0.0

    normalized_persistences = non_zero_persistences / total_persistence

    # Calculate entropy using log2
    # Add a small epsilon to avoid log(0) in case of numerical issues, though not expected with the previous check
    entropy = -np.sum(normalized_persistences * np.log2(normalized_persistences + 1e-10))
    return entropy
    
class_list = [
    ("Human", "RUES CM", "human/RUES/RUES_CM"),
    ("Human", "RUES CP", "human/RUES/RUES_CP"),
    ("Human", "RUES ESC", "human/RUES/RUES_ESC"),
    ("Human", "RUES Fetal Heart", "human/RUES/RUES_Fetal_Heart"),
    ("Human", "RUES MES", "human/RUES/RUES_MES"),
    ("Human", "WTC CM", "human/WTC/WTC_CM"),
    ("Human", "WTC CP", "human/WTC/WTC_CP"),
    ("Human", "WTC MES", "human/WTC/WTC_MES"),
    ("Human", "WTC PSC", "human/WTC/WTC_PSC"),
    ("Mouse", "CD4_Q1", "mouse/CD4_Q1"),
    ("Mouse", "CD8", "mouse/CD8"),
    ("Mouse", "DN", "mouse/DN"),
    ("Mouse", "DP", "mouse/DP"),
    ("Mouse", "TCon1", "mouse/TCon1"),
    ("Mouse", "TReg1", "mouse/TReg1"),
]

In [16]:
data = []
for species, class_name, folder_path in tqdm.tqdm(class_list, desc="Processing classes"):
    files = get_files_from_folder(folder_path)
    for file in tqdm.tqdm(files, desc=f"Processing files in {class_name}", leave=False):
        diagram = load_persistence_diagram(file)
        entropy = calculate_persistent_entropy(diagram)
        data.append({'diagram': diagram, 'entropy': entropy, 'label': species})

df = pd.DataFrame(data)
df['label'] = df['label'].map({'Human': 0, 'Mouse': 1})

X = df[['diagram', 'entropy']]
y = df['label']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

display(df.head())

Processing classes: 100%|███████████████████████| 15/15 [00:07<00:00,  2.11it/s]


,diagram,entropy,label
0,"[(0.3437791343989705, 0.3480882638718497), (0....",9.053992,0
1,"[(0.3104112007230151, 0.3303933517986081), (0....",9.105383,0
2,"[(0.4025242093635091, 0.4035859293900898), (0....",7.564137,0
3,"[(0.3304349598741515, 0.3389237699236216), (0....",7.276258,0
4,"[(0.3153011343641698, 0.3263009335749689), (0....",8.219333,0


In [17]:
class DiagramTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # X is expected to be a DataFrame with a 'diagram' column
        return X['diagram'].tolist()

class EntropyTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # X is expected to be a DataFrame with an 'entropy' column
        # Return as a NumPy array and reshape to a column vector
        return X['entropy'].values.reshape(-1, 1)

In [18]:
feature_combinations = {
    "All Features": ["PersistenceImage", "Landscape", "PersistentEntropy"],
    "Only Image": ["PersistenceImage"],
    "Only Landscape": ["Landscape"],
    "Only Entropy": ["PersistentEntropy"],
    "Image + Landscape": ["PersistenceImage", "Landscape"],
    "Image + Entropy": ["PersistenceImage", "PersistentEntropy"],
    "Landscape + Entropy": ["Landscape", "PersistentEntropy"]
}

In [ ]:
xgb_results = {}

# Define the base transformers with names
base_transformers = {
    "PersistenceImage": Pipeline([
        ("Selector", DiagramTransformer()),
        ("DiagramSeparator", gudhi.representations.DiagramSelector(limit=np.inf, point_type="finite")),
        ("Scaler", gudhi.representations.DiagramScaler(scalers=[([0,1], MinMaxScaler())])),
        ("PersistenceImage", gudhi.representations.PersistenceImage(resolution=[30, 30], bandwidth=1.0))
    ]),
    "Landscape": Pipeline([
        ("Selector", DiagramTransformer()),
        ("DiagramSeparator", gudhi.representations.DiagramSelector(limit=np.inf, point_type="finite")),
        ("Scaler", gudhi.representations.DiagramScaler(scalers=[([0,1], MinMaxScaler())])),
        ("Landscape", gudhi.representations.Landscape(resolution=100))
    ]),
    "PersistentEntropy": EntropyTransformer()
}

# Define the parameter grid for XGBoost
xgb_param_grid = {
    'xgb__n_estimators': [100, 200, 300],
    'xgb__learning_rate': [0.01, 0.1, 0.2],
    'xgb__max_depth': [3, 5, 7]
}

for combination_name, transformers_to_include in tqdm.tqdm(feature_combinations.items(), desc="XGBoost Ablation Study"):
    print(f"\nProcessing feature combination: {combination_name}")

    # Dynamically create FeatureUnion based on the current combination
    current_transformers = [(name, base_transformers[name]) for name in transformers_to_include]
    feature_union_subset = FeatureUnion(current_transformers)

    # Create the XGBoost pipeline
    xgb_pipeline = Pipeline([
        ('features', feature_union_subset),
        ('xgb', XGBClassifier(objective='binary:logistic', eval_metric='logloss'))
    ])

    # Perform GridSearchCV for XGBoost
    print(f"Performing GridSearchCV for XGBoost with {combination_name}...")
    xgb_grid_search = GridSearchCV(xgb_pipeline, xgb_param_grid, cv=5, scoring='accuracy', verbose=1)
    xgb_grid_search.fit(X_train, y_train)

    print(f"Best parameters for XGBoost ({combination_name}): {xgb_grid_search.best_params_}")
    print(f"Best cross-validation accuracy for XGBoost ({combination_name}): {xgb_grid_search.best_score_}")

    # Store the grid search object
    xgb_results[combination_name] = xgb_grid_search

XGBoost Ablation Study:   0%|                             | 0/7 [00:00<?, ?it/s]


Processing feature combination: All Features
Performing GridSearchCV for XGBoost with All Features...
Fitting 5 folds for each of 27 candidates, totalling 135 fits


XGBoost Ablation Study:  14%|██▍              | 1/7 [21:47<2:10:42, 1307.02s/it]

Best parameters for XGBoost (All Features): {'xgb__learning_rate': 0.1, 'xgb__max_depth': 3, 'xgb__n_estimators': 100}
Best cross-validation accuracy for XGBoost (All Features): 0.9839183673469387

Processing feature combination: Only Image
Performing GridSearchCV for XGBoost with Only Image...
Fitting 5 folds for each of 27 candidates, totalling 135 fits


In [ ]:
rf_results = {}

# Define the parameter grid for Random Forest
rf_param_grid = {
    'rf__n_estimators': [100, 200, 300],
    'rf__max_depth': [None, 10, 20],
    'rf__min_samples_split': [2, 5, 10]
}

for combination_name, transformers_to_include in tqdm.tqdm(feature_combinations.items(), desc="Random Forest Ablation Study"):
    print(f"\nProcessing feature combination: {combination_name}")

    # Dynamically create FeatureUnion based on the current combination
    current_transformers = [(name, base_transformers[name]) for name in transformers_to_include]
    feature_union_subset = FeatureUnion(current_transformers)

    # Create the Random Forest pipeline
    rf_pipeline = Pipeline([
        ('features', feature_union_subset),
        ('rf', RandomForestClassifier(random_state=42))
    ])

    # Perform GridSearchCV for Random Forest
    print(f"Performing GridSearchCV for Random Forest with {combination_name}...")
    rf_grid_search = GridSearchCV(rf_pipeline, rf_param_grid, cv=5, scoring='accuracy', verbose=1)
    rf_grid_search.fit(X_train, y_train)

    print(f"Best parameters for Random Forest ({combination_name}): {rf_grid_search.best_params_}")
    print(f"Best cross-validation accuracy for Random Forest ({combination_name}): {rf_grid_search.best_score_}")

    # Store the grid search object
    rf_results[combination_name] = rf_grid_search

In [ ]:
xgb_results_summary = {}

for combination_name, grid_search_result in xgb_results.items():
    best_estimator = grid_search_result.best_estimator_
    y_pred = best_estimator.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    xgb_results_summary[combination_name] = {
        'accuracy': accuracy,
        'f1_score': f1,
        'precision': precision,
        'recall': recall,
        'best_params': grid_search_result.best_params_
    }

display(xgb_results_summary)

In [ ]:
rf_results_summary = {}

for combination_name, grid_search_result in rf_results.items():
    best_estimator = grid_search_result.best_estimator_
    y_pred = best_estimator.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    rf_results_summary[combination_name] = {
        'accuracy': accuracy,
        'f1_score': f1,
        'precision': precision,
        'recall': recall,
        'best_params': grid_search_result.best_params_
    }

display(rf_results_summary)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Create a DataFrame from the xgb_results_summary dictionary
xgb_results_df = pd.DataFrame.from_dict(xgb_results_summary, orient='index')

# Reset the index and rename the index column
xgb_results_df = xgb_results_df.reset_index()
xgb_results_df = xgb_results_df.rename(columns={'index': 'Feature Combination'})

# Melt the DataFrame to long format
xgb_results_melted = xgb_results_df.melt(id_vars='Feature Combination', value_vars=['accuracy', 'f1_score', 'precision', 'recall'],
                                         var_name='Metric', value_name='Score')

# Create a bar plot
plt.figure(figsize=(12, 6))
sns.barplot(x='Feature Combination', y='Score', hue='Metric', data=xgb_results_melted)

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, ha='right')

# Add a title to the plot
plt.title('XGBoost Model Performance by Feature Combination')

# Display the plot
plt.tight_layout()
plt.show()

In [ ]:
# Create a DataFrame from the rf_results_summary dictionary
rf_results_df = pd.DataFrame.from_dict(rf_results_summary, orient='index')

# Reset the index and rename the index column
rf_results_df = rf_results_df.reset_index()
rf_results_df = rf_results_df.rename(columns={'index': 'Feature Combination'})

# Melt the DataFrame to long format
rf_results_melted = rf_results_df.melt(id_vars='Feature Combination', value_vars=['accuracy', 'f1_score', 'precision', 'recall'],
                                       var_name='Metric', value_name='Score')

# Create a bar plot
plt.figure(figsize=(12, 6))
sns.barplot(x='Feature Combination', y='Score', hue='Metric', data=rf_results_melted)

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, ha='right')

# Add a title to the plot
plt.title('Random Forest Model Performance by Feature Combination')

# Adjust plot layout
plt.tight_layout()

# Display the plot
plt.show()

In [ ]:
# Create DataFrames from the summary dictionaries
xgb_results_df = pd.DataFrame.from_dict(xgb_results_summary, orient='index').reset_index().rename(columns={'index': 'Feature Combination'})
rf_results_df = pd.DataFrame.from_dict(rf_results_summary, orient='index').reset_index().rename(columns={'index': 'Feature Combination'})

# Add a 'Model' column
xgb_results_df['Model'] = 'XGBoost'
rf_results_df['Model'] = 'Random Forest'

# Concatenate the two DataFrames
combined_results_df = pd.concat([xgb_results_df, rf_results_df])

# Melt the concatenated DataFrame
combined_results_melted = combined_results_df.melt(id_vars=['Model', 'Feature Combination'],
                                                   value_vars=['accuracy', 'f1_score', 'precision', 'recall'],
                                                   var_name='Metric', value_name='Score')

# Create a bar plot to compare performance using catplot for faceting
g = sns.catplot(x='Feature Combination', y='Score', hue='Metric', col='Model', data=combined_results_melted, kind='bar', height=6, aspect=0.8)

# Rotate x-axis labels for better readability
g.set_xticklabels(rotation=45, ha='right')

# Add a title to the plot
g.fig.suptitle('Comparison of XGBoost and Random Forest Model Performance by Feature Combination and Metric', y=1.02)

# Adjust plot layout
plt.tight_layout()

# Display the plot
plt.show()

In [ ]:
display(combined_results_df)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Function to plot confusion matrix
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()

# Generate confusion matrices for XGBoost results
print("Generating Confusion Matrices for XGBoost:")
for combination_name, grid_search_result in xgb_results.items():
    best_estimator = grid_search_result.best_estimator_
    y_pred = best_estimator.predict(X_test)
    plot_confusion_matrix(y_test, y_pred, f'XGBoost - {combination_name} Confusion Matrix')

# Generate confusion matrices for Random Forest results
print("\nGenerating Confusion Matrices for Random Forest:")
for combination_name, grid_search_result in rf_results.items():
    best_estimator = grid_search_result.best_estimator_
    y_pred = best_estimator.predict(X_test)
    plot_confusion_matrix(y_test, y_pred, f'Random Forest - {combination_name} Confusion Matrix')